In [0]:
%pip uninstall -y psycopg2 psycopg2-binary

In [0]:
%pip install -q 'databricks-sdk>=0.118.0' sentence-transformers
dbutils.library.restartPython()


In [0]:
# Define las variables de entorno y haz los imports después del último reinicio de Python.

import os

os.environ["LAKEBASE_HOST"] = "ep-sparkling-smoke-d87sujf7.database.us-east-2.cloud.databricks.com"
os.environ["LAKEBASE_PORT"] = "5432"
os.environ["LAKEBASE_DB"] = "databricks_postgres"
os.environ["LAKEBASE_USER"] = "felixemilio9312@gmail.com"
os.environ["LAKEBASE_ENDPOINT"] = "projects/research-copilot/branches/production/endpoints/primary"
os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"

from src.embeddings.embed_papers import embed_pending_papers
embed_pending_papers()

In [0]:
import json
from src.db.connection import get_connection
from src.embeddings.embed_papers import embed_query
 
vec = embed_query("aprender los fundamentos de RAG en LLMs")
 
with get_connection() as conn, conn.cursor() as cur:
    cur.execute("SELECT * FROM search_papers(%s::vector, %s)", (json.dumps(vec), 5))
    for row in cur.fetchall():
        print(row["title"])

In [0]:
from src.db.connection import get_connection

with get_connection() as conn, conn.cursor() as cur:
    cur.execute("SELECT count(*) AS n FROM papers")
    print("Total papers:", cur.fetchone()["n"])

    cur.execute("SELECT count(*) AS n FROM papers WHERE embedding IS NOT NULL")
    print("Papers con embedding:", cur.fetchone()["n"])

    cur.execute("SELECT count(*) AS n FROM papers WHERE abstract IS NULL OR abstract = ''")
    print("Papers sin abstract (se saltan al embeber):", cur.fetchone()["n"])

    cur.execute("SELECT paper_id, title FROM papers WHERE embedding IS NOT NULL LIMIT 3")
    print("Muestra con embedding:", cur.fetchall())

    cur.execute("""
        SELECT proname, pg_get_function_identity_arguments(oid) AS args
        FROM pg_proc
        WHERE proname = 'search_papers'
    """)
    print("Firma de search_papers en Postgres:", cur.fetchall())

In [0]:
from src.embeddings.embed_papers import embed_pending_papers
embed_pending_papers(batch_size=50)

In [0]:
from src.db.connection import get_connection

SP_CLIENT_ID = "f18aa387-603a-407e-aa55-0360863279a6"  # el nuevo, no el viejo

with get_connection() as conn, conn.cursor() as cur:
    cur.execute(f'GRANT USAGE ON SCHEMA public TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON authors TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON paper_authors TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON collection_papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT ON reading_progress TO "{SP_CLIENT_ID}"')
    conn.commit()
print("Grants aplicados al nuevo SP")

In [0]:
from src.db.connection import get_connection

SP_CLIENT_ID = "f18aa387-603a-407e-aa55-0360863279a6"

with get_connection() as conn, conn.cursor() as cur:
    cur.execute(f'GRANT USAGE ON SCHEMA public TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON authors TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON paper_authors TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON collection_papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT ON reading_progress TO "{SP_CLIENT_ID}"')
    conn.commit()

# Verificacion inmediata
cur = conn.cursor()
with get_connection() as conn, conn.cursor() as cur:
    cur.execute("""
        SELECT grantee, table_name, privilege_type
        FROM information_schema.role_table_g
        WHERE grantee = %s
        ORDER BY table_name
    """, (SP_CLIENT_ID,))
    for row in cur.fetchall():
        print(row)

In [0]:
from src.db.connection import get_connection

SP_CLIENT_ID = "f18aa387-603a-407e-aa55-0360863279a6"

with get_connection() as conn, conn.cursor() as cur:
    cur.execute(f'GRANT UPDATE ON papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT UPDATE ON authors TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT UPDATE ON paper_authors TO "{SP_CLIENT_ID}"')
    conn.commit()
print("UPDATE grants aplicados")

In [0]:
from src.db.connection import get_connection

SP_CLIENT_ID = "f18aa387-603a-407e-aa55-0360863279a6"

with get_connection() as conn, conn.cursor() as cur:
    cur.execute("SELECT grantee, table_name, privilege_type FROM information_schema.role_table_grants WHERE grantee = %s ORDER BY table_name", (SP_CLIENT_ID,))
    for row in cur.fetchall():
        print(row)

In [0]:
from src.db.connection import get_connection

with get_connection() as conn, conn.cursor() as cur:
    cur.execute("DROP FUNCTION search_papers(vector, integer)")
    cur.execute("""
        CREATE FUNCTION search_papers(goal_embedding vector(384), top_k INT DEFAULT 6)
        RETURNS TABLE (paper_id TEXT, title TEXT, abstract TEXT, doi TEXT, oa_url TEXT, distance FLOAT)
        LANGUAGE SQL AS $$
          SELECT paper_id, title, abstract, doi, oa_url, embedding <=> goal_embedding AS distance
          FROM papers
          WHERE embedding IS NOT NULL
          ORDER BY distance
          LIMIT top_k;
        $$;
    """)
    conn.commit()
print("search_papers recreada con columna distance")

In [0]:
import json
from src.db.connection import get_connection
from src.embeddings.embed_papers import embed_query

for q in ["sistemas de machine learning", "machine learning systems",
          "sistemas RAG", "retrieval augmented generation"]:
    vec = embed_query(q)
    with get_connection() as conn, conn.cursor() as cur:
        cur.execute("SELECT * FROM search_papers(%s::vector, %s)", (json.dumps(vec), 5))
        print("===", q)
        for r in cur.fetchall():
            print("  ", round(r["distance"], 3), "-", r["title"][:80])

In [0]:
from src.db.connection import get_connection

SP_CLIENT_ID = "7dca60d0-f0eb-462c-8326-037ccb1e82f1"

with get_connection() as conn, conn.cursor() as cur:
    cur.execute(f'GRANT USAGE ON SCHEMA public TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON users TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT ON collections TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT, DELETE ON collection_papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT ON papers TO "{SP_CLIENT_ID}"')
    cur.execute(f'GRANT SELECT, INSERT, UPDATE ON reading_progress TO "{SP_CLIENT_ID}"')
    conn.commit()
print("Grants aplicados")

In [0]:
from src.db.connection import get_connection

junk_ids = [
    "openalex:W7202148741",  # El Subestimado...
    "openalex:W7202143324",  # El Subestimado... (duplicado)
    "openalex:W7202182318",  # Aplicacao de LLM com RAG (PT)
    "openalex:W2905538815",  # Qualidade da Atencao Primaria (PT)
    "openalex:W7202068505",  # USO DO CHATGPT-4 (PT)
    "openalex:W7202133850",  # Archivo Maestro Visionario Cuantico
]

with get_connection() as conn, conn.cursor() as cur:
    cur.execute("DELETE FROM papers WHERE paper_id = ANY(%s)", (junk_ids,))
    print("Borrados por id:", cur.rowcount)
    cur.execute("DELETE FROM papers WHERE title ILIKE %s", ("%CRÁTON%",))
    print("Borrados por titulo (craton):", cur.rowcount)
    conn.commit()

    cur.execute("SELECT count(*) AS total FROM papers")
    print("Total restante:", cur.fetchone())